# Load S3 Parquet data into Redshift with COPY and pick the right distribution key

The reporting team's BI dashboard times out on the curated layer because Athena cannot meet their 1-second SLA on three representative queries. You have one session to stand up a Redshift cluster, load the Parquet via COPY, pick the right distribution and sort keys, and prove the dashboard SLA holds before the 9 AM exec review.

The literal scenario from the cert YAML is 50 million order rows. This lab uses a seeded dataset of about 50,000 rows so cluster create plus load plus queries fit inside the 65-minute session window. The exam pattern and the optimization mechanics hold at both scales; the only adjustment is row count.

The four deliverables map to four checkpoints:

1. A single-node `dc2.large` Redshift cluster in the default VPC's public subnet with an IP-locked security group, the COPY role attached, and the cluster registered as critical in `CLEANUP_MANIFEST` before the waiter polls.
2. The Parquet orders dataset loaded via `COPY ... FORMAT AS PARQUET`, row count matches the seed, and `STL_LOAD_ERRORS` reports zero rows for the COPY.
3. The orders table rebuilt with `DISTSTYLE KEY`, `DISTKEY(customer_id)`, and `SORTKEY(order_date)`; `EXPLAIN` on the join-heavy query no longer reports `DS_BCAST_INNER` or `DS_DIST_BOTH`.
4. Three representative queries (filter-by-customer, date-range aggregation, join-against-dimension) each return server-side under 1000 ms via the redshift-data API `Duration` field.

**Time.** About 65 minutes hands-on. The cluster waiter (4 to 6 minutes) and the cleanup waiter (similar) dominate wall-clock; the writing is short.

**Cost.** HOURLY METER WARNING. This is the first lab in the DEA-C01 sequence that creates an hourly-billed resource. Redshift `dc2.large` bills $0.25 per hour on demand: $6 per day, $42 per week if you forget about it. A typical session is well under an hour, so the happy-path cost is $0.15 to $0.25. The 2-hour EventBridge plus Lambda safety net guarantees the cluster auto-destroys at the 2-hour wall-clock mark even if your kernel dies; worst case is $0.50 per session. Three independent failures (cleanup cell skipped, atexit fails, safety-net Lambda fails) would have to compound to leave the cluster running overnight. Even so: set the $20 per month billing alert in your AWS console before you start. Nobody plans for the three-things-fail case.

This lab maps to AWS DEA-C01 Domain 2: Data Store Management (26% of exam weight).


In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.3.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT
# section 12. Lab 06 is the first critical-resource lab in the DEA-C01
# sequence; the safety-net Lambda plus EventBridge schedule rule are
# declared here so the cluster cannot outlive the 2-hour wall-clock window.

import atexit
import getpass
import io
import json
import random
import string
import sys
import time
import urllib.request
from datetime import datetime, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "redshift-loading-and-queries"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[5].slug exactly
REGION = "us-east-1"  # all DEA-C01 labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource names. Bucket name appends the account ID for global uniqueness.
CLUSTER_ID = f"labrun-{LAB_ID}"
SG_NAME = f"labrun-{LAB_ID}-sg"
COPY_ROLE_NAME = f"labrun-{LAB_ID}-copy-role"
SAFETY_NET_LAMBDA_NAME = f"labrun-{LAB_ID}-safety-net"
SAFETY_NET_RULE_NAME = f"labrun-{LAB_ID}-safety-net-rule"
SAFETY_NET_ROLE_NAME = f"labrun-{LAB_ID}-safety-net-role"
BUCKET_NAME = None  # resolved after STS once the account ID is known

# Cluster shape. dc2.large is the cheapest provisioned node type ($0.25/hour)
# and matches the cert-canonical exam shape. RA3 supports ALTER DISTKEY;
# dc2.large does not, so Checkpoint 3 walks the drop-and-recreate path.
NODE_TYPE = "dc2.large"
NUMBER_OF_NODES = 1
DB_NAME = "labrun"
DB_USER = "labrun_admin"
DB_PORT = 5439

# Generated admin password. Redshift requires uppercase, lowercase, and a
# digit. The password is held in memory only; the redshift-data API uses
# the temporary credentials path via DbUser, not this password.
_pw_chars = string.ascii_letters + string.digits
_rng = random.Random(0xC0FFEE)
DB_PASSWORD = "Lab" + "".join(_rng.choice(_pw_chars) for _ in range(20)) + "9"

# Seed config. Deterministic so row counts and Parquet bytes are stable
# across student runs. About 50,000 orders rows; small enough for COPY to
# finish in under a minute on dc2.large, large enough to exercise distkey
# and sortkey behaviour on the EXPLAIN plans.
SEED = 42
ORDERS_ROWS = 50_000
CUSTOMERS_ROWS = 500

# Safety-net schedule. The cluster auto-destroys 2 hours after setup runs
# per RESOURCE-SAFETY-SPEC Section 3 Lab 6 mitigation. The EventBridge
# schedule expression is computed at setup time relative to wall clock.
SAFETY_NET_HOURS = 2

# Module-level cache for Checkpoint 4. Each query duration in milliseconds
# is recorded here by Task 4 so the checkpoint validator does not have to
# rerun the queries.
QUERY_DURATIONS_MS: dict[str, int] = {}


In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first Redshift call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve the bucket name now that we know the account ID. S3 bucket names
# must be globally unique.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
#
# Per RESOURCE-SAFETY-SPEC Rule 2, the cluster and the safety-net pair
# (EventBridge schedule rule plus Lambda) tear down FIRST because they are
# the hourly-billed resources. Standard resources (safety-net role, SG,
# COPY role, S3 bucket) come after. The manifest below lists every
# resource the lab creates in strict reverse-creation, critical-first
# order so the atexit handler tears down the cluster even if the kernel
# dies during the create_cluster waiter.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# labrun-checks 0.3.0 lacks adapters for redshift_cluster,
# eventbridge_rule, lambda_function, and security_group. The cleanup cell
# at the bottom of this notebook defines an inline _LabAdapter that wraps
# AwsCleanupAdapter and dispatches those four types inline. The new
# resource types are still declared declaratively here so the canonical
# summary, atexit handler, and orphan scan all see them.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="redshift_cluster",
        id=CLUSTER_ID,
        region=REGION,
        cli_delete_command=(
            f"aws redshift delete-cluster --cluster-identifier {CLUSTER_ID} "
            f"--skip-final-cluster-snapshot"
        ),
        critical=True,
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=SAFETY_NET_RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {SAFETY_NET_RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {SAFETY_NET_RULE_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="lambda_function",
        id=SAFETY_NET_LAMBDA_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws lambda delete-function --function-name {SAFETY_NET_LAMBDA_NAME}"
        ),
        critical=True,
    ),
    CleanupResource(
        type="iam_role",
        id=SAFETY_NET_ROLE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws iam delete-role --role-name {SAFETY_NET_ROLE_NAME}"
        ),
    ),
    CleanupResource(
        type="security_group",
        id=SG_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws ec2 delete-security-group --group-name {SG_NAME}"
        ),
    ),
    CleanupResource(
        type="iam_role",
        id=COPY_ROLE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws iam delete-role --role-name {COPY_ROLE_NAME}"
        ),
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is
    the safety net for kernel crashes during the create_cluster waiter or
    any other long-running step. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter()) if "_LabAdapter" in globals() else run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for
    leftover state with the lab's tag and surface the cleanup command
    before creating any new resources. This matters extra for Lab 06
    because a leftover cluster keeps billing $0.25 per hour.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources")
    print("or, worse, a second Redshift cluster billing in parallel. Run")
    print("the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints,")
    print("then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Task 1: Provision the Redshift cluster and register it as critical before the waiter polls

This is the hottest cell in any DEA-C01 lab. The moment `create_cluster` returns, the meter starts at $0.25 per hour. The order of operations in this cell matters: the cluster goes into `CLEANUP_MANIFEST` immediately after `create_cluster` returns, BEFORE the waiter starts polling. If the kernel dies during the 4-to-6 minute waiter, the atexit handler still has the manifest entry and can delete the cluster.

Five steps in this cell, in order:

1. **Create the S3 bucket and the COPY role.** The bucket holds the seeded Parquet under `orders/` and `customers/`. The COPY role is the IAM role Redshift assumes to read from S3; it carries `AmazonS3ReadOnlyAccess` plus a trust policy that lets `redshift.amazonaws.com` assume it. The COPY role ARN is passed to `create_cluster` via `IamRoles=[...]` so the COPY command can reference it without inline credentials.
2. **Create the security group locked to your current public IP.** Port 5439 inbound. The IP comes from `https://checkip.amazonaws.com` (an AWS-owned endpoint; no third-party fetch). If you move networks mid-session, queries will start failing; Hint 3 covers the IP refresh path.
3. **Create the safety-net Lambda and EventBridge schedule rule.** The Lambda has one job: at the 2-hour wall-clock mark, delete the cluster but ONLY if its tags confirm `labrun:lab-slug == redshift-loading-and-queries`. The tag-scoped delete is the seatbelt that prevents a misconfigured safety net from deleting an unrelated cluster.
4. **Call `redshift.create_cluster`.** Single node `dc2.large`, `PubliclyAccessible=True`, the lab SG attached, the COPY role attached via `IamRoles`. Append the cluster entry to `CLEANUP_MANIFEST` (or update the existing entry if you prefer; the manifest cell above already pre-declared it as critical so atexit catches a kernel crash during the waiter).
5. **Wait for the cluster to become `available`.** Typical wait is 4 to 6 minutes. The waiter does not change the cost story; the meter has been running since step 4.

Checkpoint 1 confirms the cluster is `available`, the node type and node count are correct, `PubliclyAccessible` is `True`, the lab SG is attached, and the cluster appears in `CLEANUP_MANIFEST` with `critical=True`.


In [ ]:
# NBVAL_SKIP
# Task 1: Bucket, COPY role, security group, safety-net Lambda plus
# EventBridge rule, then create the Redshift cluster and wait for it to
# become available. The order below is deliberate; the cluster is the LAST
# thing created so the safety net is already in place when the meter starts.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
ec2 = boto3.client(
    "ec2",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
redshift = boto3.client(
    "redshift",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# 1. Bucket. us-east-1 rejects LocationConstraint; other regions require it.
# YOUR CODE: s3.create_bucket(Bucket=BUCKET_NAME).
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

# 2. COPY role. Trust policy lets redshift.amazonaws.com assume; permission
# policy is read on the lab bucket only. Inline policy with s3:GetObject
# plus s3:ListBucket on the bucket and bucket/*. The Checkpoint 1 validator
# does not inspect this role directly; the COPY command in Task 2 will fail
# if the role is missing or under-privileged.
copy_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "redshift.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
copy_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:ListBucket"],
        "Resource": [
            f"arn:aws:s3:::{BUCKET_NAME}",
            f"arn:aws:s3:::{BUCKET_NAME}/*",
        ],
    }],
}
# YOUR CODE: iam.create_role(
#   RoleName=COPY_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(copy_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=COPY_ROLE_NAME,
#   PolicyName="labrun-copy-policy",
#   PolicyDocument=json.dumps(copy_inline_policy),
# ).
copy_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}"
print(f"COPY role ARN: {copy_role_arn}")

# 3. Security group locked to the caller's public IP. Port 5439 inbound.
my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()
my_cidr = f"{my_ip}/32"
default_vpc = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])
default_vpc_id = default_vpc["Vpcs"][0]["VpcId"]
sg_resp = ec2.create_security_group(
    GroupName=SG_NAME,
    Description=f"Redshift access for {LAB_ID}, locked to caller IP",
    VpcId=default_vpc_id,
    TagSpecifications=[{
        "ResourceType": "security-group",
        "Tags": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    }],
)
sg_id = sg_resp["GroupId"]
ec2.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[{
        "IpProtocol": "tcp",
        "FromPort": DB_PORT,
        "ToPort": DB_PORT,
        "IpRanges": [{"CidrIp": my_cidr, "Description": "labrun lab caller IP"}],
    }],
)
print(f"Security group created: {sg_id} ({SG_NAME}); ingress 5439 from {my_cidr}")

# 4. Safety-net IAM role + Lambda + EventBridge rule.
sn_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
sn_inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "redshift:DescribeClusters",
                "redshift:DeleteCluster",
                "redshift:DescribeTags",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
    ],
}
# YOUR CODE: iam.create_role(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: iam.put_role_policy(
#   RoleName=SAFETY_NET_ROLE_NAME,
#   PolicyName="labrun-safety-net-policy",
#   PolicyDocument=json.dumps(sn_inline_policy),
# ).
sn_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{SAFETY_NET_ROLE_NAME}"

# Safety-net Lambda source. Tag-scoped delete: the Lambda confirms the
# cluster's labrun:lab-slug tag matches this lab before calling
# delete-cluster, so a misconfigured safety net cannot delete an unrelated
# cluster. SkipFinalClusterSnapshot=True so delete returns fast and cannot
# block on snapshot.
lambda_source = f'''
import boto3

CLUSTER_ID = "{CLUSTER_ID}"
EXPECTED_TAG_KEY = "{LAB_TAG_KEY}"
EXPECTED_TAG_VALUE = "{LAB_TAG_VALUE}"


def handler(event, context):
    rs = boto3.client("redshift")
    try:
        resp = rs.describe_clusters(ClusterIdentifier=CLUSTER_ID)
    except rs.exceptions.ClusterNotFoundFault:
        print(f"Cluster {{CLUSTER_ID}} not found; nothing to delete.")
        return {{"status": "noop"}}
    cluster = resp["Clusters"][0]
    tags = {{t["Key"]: t["Value"] for t in cluster.get("Tags", [])}}
    if tags.get(EXPECTED_TAG_KEY) != EXPECTED_TAG_VALUE:
        print(
            f"Refusing to delete {{CLUSTER_ID}}: tag {{EXPECTED_TAG_KEY}} is "
            f"{{tags.get(EXPECTED_TAG_KEY)!r}}, expected {{EXPECTED_TAG_VALUE!r}}."
        )
        return {{"status": "refused"}}
    rs.delete_cluster(
        ClusterIdentifier=CLUSTER_ID,
        SkipFinalClusterSnapshot=True,
    )
    print(f"Cluster {{CLUSTER_ID}} delete initiated by safety net.")
    return {{"status": "deleted"}}
'''

# Zip the source in memory and create the Lambda function. boto3 will
# accept ZipFile bytes inline so no S3 stage is needed.
import zipfile
zbuf = io.BytesIO()
with zipfile.ZipFile(zbuf, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("index.py", lambda_source)
zip_bytes = zbuf.getvalue()

# IAM role propagation can take a few seconds before Lambda will accept it.
# Retry the create_function call to ride out the propagation delay.
_lambda_deadline = time.time() + 60
while True:
    try:
        # YOUR CODE: lambda_client.create_function(
        #   FunctionName=SAFETY_NET_LAMBDA_NAME,
        #   Runtime="python3.11",
        #   Role=sn_role_arn,
        #   Handler="index.handler",
        #   Code={"ZipFile": zip_bytes},
        #   Timeout=60,
        #   Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
        # ).
        break
    except ClientError as e:
        if e.response["Error"]["Code"] == "InvalidParameterValueException" and time.time() < _lambda_deadline:
            time.sleep(5)
            continue
        raise
lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{SAFETY_NET_LAMBDA_NAME}"
print(f"Safety-net Lambda created: {lambda_arn}")

# EventBridge schedule rule: fire once at lab_start + SAFETY_NET_HOURS.
# cron(minutes hours day month ? year). Use a one-shot expression at the
# computed UTC wall-clock instant. timedelta handles day/month/year rollover
# so a session that straddles midnight UTC still schedules correctly.
from datetime import timedelta
fire_at = (datetime.now(timezone.utc) + timedelta(hours=SAFETY_NET_HOURS)).replace(microsecond=0)
cron_expr = (
    f"cron({fire_at.minute} {fire_at.hour} "
    f"{fire_at.day} {fire_at.month} ? {fire_at.year})"
)
# YOUR CODE: events.put_rule(
#   Name=SAFETY_NET_RULE_NAME,
#   ScheduleExpression=cron_expr,
#   State="ENABLED",
#   Description=f"Safety net for {LAB_ID}; auto-deletes cluster at +{SAFETY_NET_HOURS}h.",
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
# YOUR CODE: lambda_client.add_permission(
#   FunctionName=SAFETY_NET_LAMBDA_NAME,
#   StatementId="labrun-eventbridge-invoke",
#   Action="lambda:InvokeFunction",
#   Principal="events.amazonaws.com",
#   SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
# ).
# YOUR CODE: events.put_targets(
#   Rule=SAFETY_NET_RULE_NAME,
#   Targets=[{"Id": "1", "Arn": lambda_arn}],
# ).
print(f"EventBridge rule {SAFETY_NET_RULE_NAME} scheduled at {cron_expr}")

# 5. Create the Redshift cluster. The manifest entry was pre-declared in
# the manifest cell above with critical=True, so atexit catches a kernel
# crash during the waiter even though the cluster is not yet "available".
# YOUR CODE: redshift.create_cluster(
#   ClusterIdentifier=CLUSTER_ID,
#   NodeType=NODE_TYPE,
#   NumberOfNodes=NUMBER_OF_NODES,
#   ClusterType="single-node",
#   MasterUsername=DB_USER,
#   MasterUserPassword=DB_PASSWORD,
#   DBName=DB_NAME,
#   PubliclyAccessible=True,
#   VpcSecurityGroupIds=[sg_id],
#   IamRoles=[copy_role_arn],
#   Port=DB_PORT,
#   Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
# ).
print(f"create_cluster returned for {CLUSTER_ID}; meter has started at $0.25/hour.")
print("Waiting for cluster to become available (4 to 6 minutes typical)...")

waiter = redshift.get_waiter("cluster_available")
waiter.wait(
    ClusterIdentifier=CLUSTER_ID,
    WaiterConfig={"Delay": 30, "MaxAttempts": 30},
)
print(f"Cluster {CLUSTER_ID} is available.")


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Redshift cluster is available, node shape matches,
# publicly accessible with the lab SG attached, and present in
# CLEANUP_MANIFEST with critical=True.


def checkpoint_1(session):
    try:
        redshift_client = boto3.client(
            "redshift",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            resp = redshift_client.describe_clusters(ClusterIdentifier=CLUSTER_ID)
        except ClientError as e:
            code_ = e.response["Error"]["Code"]
            if code_ in ("ClusterNotFound", "ClusterNotFoundFault"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Redshift cluster {CLUSTER_ID!r} does not exist. "
                        f"Run the Task 1 cell to create it."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        clusters = resp.get("Clusters", [])
        if not clusters:
            return CheckpointResult(
                status="fail",
                error_reason=f"describe_clusters returned no clusters for {CLUSTER_ID!r}.",
            )
        cluster = clusters[0]

        status_ = cluster.get("ClusterStatus", "")
        if status_ != "available":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster status is {status_!r}, expected 'available'. "
                    f"Wait for the create_cluster waiter to finish, then re-run "
                    f"this checkpoint."
                ),
            )

        node_type = cluster.get("NodeType", "")
        if node_type != NODE_TYPE:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NodeType is {node_type!r}, expected {NODE_TYPE!r}. "
                    f"This lab requires dc2.large for cost predictability."
                ),
            )

        number_of_nodes = cluster.get("NumberOfNodes", 0)
        if number_of_nodes != NUMBER_OF_NODES:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"NumberOfNodes is {number_of_nodes}, expected {NUMBER_OF_NODES}."
                ),
            )

        if not cluster.get("PubliclyAccessible", False):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "PubliclyAccessible is False. The lab uses the default VPC's "
                    "public subnet with an IP-locked security group; "
                    "PubliclyAccessible must be True."
                ),
            )

        sg_ids = {sg.get("VpcSecurityGroupId", "") for sg in cluster.get("VpcSecurityGroups", [])}
        ec2_client = boto3.client(
            "ec2",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        sg_resp = ec2_client.describe_security_groups(
            Filters=[{"Name": "group-name", "Values": [SG_NAME]}],
        )
        lab_sg_ids = {g["GroupId"] for g in sg_resp.get("SecurityGroups", [])}
        if not (sg_ids & lab_sg_ids):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster VpcSecurityGroupIds {sorted(sg_ids)} does not contain "
                    f"the lab SG (one of {sorted(lab_sg_ids)} for {SG_NAME!r}). "
                    f"Re-create the cluster with VpcSecurityGroupIds=[sg_id] or use "
                    f"redshift.modify_cluster to attach the lab SG."
                ),
            )

        # Module-level manifest inspection. The cluster must be present with
        # critical=True so atexit catches a kernel crash, and so the
        # companion panel idle-warning fires per RESOURCE-SAFETY-SPEC Rule 5.
        manifest = globals().get("CLEANUP_MANIFEST", [])
        cluster_entry = next(
            (r for r in manifest if r.type == "redshift_cluster" and r.id == CLUSTER_ID),
            None,
        )
        if cluster_entry is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster {CLUSTER_ID!r} is not in CLEANUP_MANIFEST. atexit "
                    f"will not be able to tear it down on kernel crash. Add a "
                    f"CleanupResource(type='redshift_cluster', id=CLUSTER_ID, "
                    f"critical=True, ...) entry to CLEANUP_MANIFEST."
                ),
            )
        if not getattr(cluster_entry, "critical", False):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cluster entry in CLEANUP_MANIFEST is missing critical=True. "
                    f"Per RESOURCE-SAFETY-SPEC Rule 2 the cluster tears down first; "
                    f"the companion panel idle-warning reads this flag (Rule 5)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks six properties in order: cluster exists, `ClusterStatus == "available"`, `NodeType == "dc2.large"`, `NumberOfNodes == 1`, `PubliclyAccessible is True`, the lab SG is in `VpcSecurityGroupIds`, and `CLEANUP_MANIFEST` contains an entry for the cluster with `critical=True`. Read the failure reason. The most common miss on the first run is forgetting `VpcSecurityGroupIds=[sg_id]` in the `create_cluster` call; the cluster lands behind the default SG and Checkpoint 1 fails on the SG check. The next most common miss is forgetting to call `iam.create_role` plus `iam.put_role_policy` for the COPY role before `create_cluster`; the cluster comes up but Task 2's COPY will fail because the role does not exist or has no S3 read.

</details>

<details><summary>Hint 2 (direction)</summary>

Five API calls in this task. `s3.create_bucket(Bucket=BUCKET_NAME)` (no `LocationConstraint` in us-east-1). `iam.create_role(RoleName=COPY_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(copy_trust_policy), Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])` plus `iam.put_role_policy(RoleName=COPY_ROLE_NAME, PolicyName="labrun-copy-policy", PolicyDocument=json.dumps(copy_inline_policy))`. Same pattern for `SAFETY_NET_ROLE_NAME`. `lambda_client.create_function(FunctionName=SAFETY_NET_LAMBDA_NAME, Runtime="python3.11", Role=sn_role_arn, Handler="index.handler", Code={"ZipFile": zip_bytes}, Timeout=60, Tags={LAB_TAG_KEY: LAB_TAG_VALUE})` inside the retry loop; IAM role propagation usually settles in under 10 seconds. `events.put_rule` plus `lambda_client.add_permission` plus `events.put_targets` wire the schedule rule to the Lambda. Finally `redshift.create_cluster(ClusterIdentifier=CLUSTER_ID, NodeType=NODE_TYPE, NumberOfNodes=NUMBER_OF_NODES, ClusterType="single-node", MasterUsername=DB_USER, MasterUserPassword=DB_PASSWORD, DBName=DB_NAME, PubliclyAccessible=True, VpcSecurityGroupIds=[sg_id], IamRoles=[copy_role_arn], Port=DB_PORT, Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])`. The `CLEANUP_MANIFEST` cluster entry was pre-declared so atexit catches kernel crash during the waiter; you do not need to append again.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
s3.create_bucket(Bucket=BUCKET_NAME)

iam.create_role(
    RoleName=COPY_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(copy_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=COPY_ROLE_NAME,
    PolicyName="labrun-copy-policy",
    PolicyDocument=json.dumps(copy_inline_policy),
)

iam.create_role(
    RoleName=SAFETY_NET_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(sn_trust_policy),
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
iam.put_role_policy(
    RoleName=SAFETY_NET_ROLE_NAME,
    PolicyName="labrun-safety-net-policy",
    PolicyDocument=json.dumps(sn_inline_policy),
)

# Inside the retry loop:
lambda_client.create_function(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    Runtime="python3.11",
    Role=sn_role_arn,
    Handler="index.handler",
    Code={"ZipFile": zip_bytes},
    Timeout=60,
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)

events.put_rule(
    Name=SAFETY_NET_RULE_NAME,
    ScheduleExpression=cron_expr,
    State="ENABLED",
    Description=f"Safety net for {LAB_ID}; auto-deletes cluster at +{SAFETY_NET_HOURS}h.",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
lambda_client.add_permission(
    FunctionName=SAFETY_NET_LAMBDA_NAME,
    StatementId="labrun-eventbridge-invoke",
    Action="lambda:InvokeFunction",
    Principal="events.amazonaws.com",
    SourceArn=f"arn:aws:events:{REGION}:{ACCOUNT_ID}:rule/{SAFETY_NET_RULE_NAME}",
)
events.put_targets(
    Rule=SAFETY_NET_RULE_NAME,
    Targets=[{"Id": "1", "Arn": lambda_arn}],
)

redshift.create_cluster(
    ClusterIdentifier=CLUSTER_ID,
    NodeType=NODE_TYPE,
    NumberOfNodes=NUMBER_OF_NODES,
    ClusterType="single-node",
    MasterUsername=DB_USER,
    MasterUserPassword=DB_PASSWORD,
    DBName=DB_NAME,
    PubliclyAccessible=True,
    VpcSecurityGroupIds=[sg_id],
    IamRoles=[copy_role_arn],
    Port=DB_PORT,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

If `create_cluster` returns `AccessDenied`, your `labrun-test` IAM user is missing one of the managed policies listed at the top of this lab (`AmazonRedshiftFullAccess`, `IAMFullAccess`, `AmazonEC2FullAccess`, `AWSLambda_FullAccess`, `AmazonEventBridgeFullAccess`). If the waiter raises `WaiterError: Max attempts exceeded`, the cluster is still creating; re-run the waiter (not `create_cluster` again) by calling `redshift.get_waiter("cluster_available").wait(ClusterIdentifier=CLUSTER_ID, ...)`. If your public IP changed mid-session, run `my_ip = urllib.request.urlopen("https://checkip.amazonaws.com").read().decode().strip()` then `ec2.authorize_security_group_ingress` (or `revoke_security_group_ingress` first if the old rule still exists) to add the new CIDR.

</details>


**Wallet check.** **Meter started.** The moment `create_cluster` returned, the Redshift `dc2.large` node started billing at $0.25 per hour. Four to six minutes for the cluster to reach `available` is about 2 to 3 cents. The 2-hour EventBridge safety net is the seatbelt: even if your kernel dies right now, the Lambda will tag-check the cluster and call `delete-cluster` at the 2-hour mark, capping worst case at $0.50. The S3 bucket, IAM roles, security group, Lambda function, and EventBridge rule are all free at this volume. Keep moving; Tasks 2 through 4 are bytes-cheap but the cluster runs the meter until cleanup deletes it.


## Task 2: Seed Parquet under `orders/` and `customers/`, then COPY into Redshift

Now the data. Four pieces in this task:

1. **Generate and upload the seed Parquet.** Two tables: `orders` (50,000 rows: `order_id`, `customer_id`, `order_date`, `amount_cents`, `status`) and `customers` (500 rows: `customer_id`, `country`, `tier`). Deterministic via `SEED`. Both written as Parquet using `pyarrow`. The orders dataset is sized so COPY finishes in under a minute on `dc2.large`; the customers dataset is the join target for Checkpoint 3 and Checkpoint 4.
2. **Create the empty tables in Redshift.** First pass: no distkey, no sortkey. Plain `CREATE TABLE` with column types. Checkpoint 3 will drop and recreate `orders` with the right distkey and sortkey; the first pass is intentionally unoptimized so the `EXPLAIN` deltas in Checkpoint 3 are visible.
3. **Run the COPY commands.** `COPY orders FROM 's3://{BUCKET}/orders/' IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;` and the same shape for `customers`. The IAM_ROLE clause references the COPY role you attached in Task 1; no inline credentials anywhere.
4. **Verify the load.** `SELECT count(*) FROM orders` returns 50,000. `SELECT count(*) FROM STL_LOAD_ERRORS WHERE filename LIKE 's3://{BUCKET}/orders/%'` returns 0.

The redshift-data API is the credential-free interface to Redshift. You pass `ClusterIdentifier`, `Database`, and `DbUser`; redshift-data uses IAM (your `labrun-test` user with `AmazonRedshiftFullAccess`) to get a temporary database credential and run the SQL. No JDBC connection string, no psql, no driver. Every query in Tasks 2 through 4 follows this pattern.

Checkpoint 2 connects via redshift-data, runs `SELECT count(*) FROM orders` and asserts the count matches `ORDERS_ROWS`, and queries `STL_LOAD_ERRORS` to assert zero rows.


In [ ]:
# NBVAL_SKIP
# Task 2: Seed Parquet, create the tables in Redshift, run COPY, verify the
# row count and the lack of STL_LOAD_ERRORS rows.

redshift_data = boto3.client(
    "redshift-data",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def run_redshift_sql(sql: str, with_result: bool = False) -> dict:
    """Submit SQL via redshift-data, poll until terminal, return describe payload.

    When with_result is True the function also fetches and attaches the
    first page of get_statement_result under key "result". Raises
    RuntimeError on FAILED or ABORTED so the cell stops on a bad query.
    """
    start = redshift_data.execute_statement(
        ClusterIdentifier=CLUSTER_ID,
        Database=DB_NAME,
        DbUser=DB_USER,
        Sql=sql,
    )
    sid = start["Id"]
    deadline = time.time() + 120
    while time.time() < deadline:
        desc = redshift_data.describe_statement(Id=sid)
        status_ = desc["Status"]
        if status_ in ("FINISHED", "FAILED", "ABORTED"):
            if status_ != "FINISHED":
                err = desc.get("Error", "(no error message)")
                raise RuntimeError(f"redshift-data {sid} ended {status_}: {err}")
            if with_result and desc.get("HasResultSet"):
                desc["result"] = redshift_data.get_statement_result(Id=sid)
            return desc
        time.sleep(1)
    raise RuntimeError(f"redshift-data {sid} did not finish within 2 minutes.")


# 1. Build the seed Parquet in memory and upload to s3://{BUCKET}/orders/
# and s3://{BUCKET}/customers/.
import pyarrow as pa
import pyarrow.parquet as pq

random.seed(SEED)
COUNTRIES = ["US", "CA", "GB", "DE", "FR", "JP", "AU"]
TIERS = ["bronze", "silver", "gold", "platinum"]
STATUSES = ["pending", "shipped", "delivered", "cancelled"]
START_DATE = datetime(2024, 1, 1)

customers_data = {
    "customer_id": list(range(1, CUSTOMERS_ROWS + 1)),
    "country": [random.choice(COUNTRIES) for _ in range(CUSTOMERS_ROWS)],
    "tier": [random.choice(TIERS) for _ in range(CUSTOMERS_ROWS)],
}
customers_table = pa.table(customers_data)
buf = io.BytesIO()
pq.write_table(customers_table, buf)
customers_bytes = buf.getvalue()
# YOUR CODE: s3.put_object(
#   Bucket=BUCKET_NAME,
#   Key="customers/customers.parquet",
#   Body=customers_bytes,
# ).
print(f"Customers Parquet uploaded ({CUSTOMERS_ROWS} rows, {len(customers_bytes)} bytes)")

# Spread order_date across 2024-01-01 to 2024-12-31 deterministically.
order_dates = []
for _ in range(ORDERS_ROWS):
    delta = random.randint(0, 364)
    d_ts = START_DATE.timestamp() + delta * 86400
    order_dates.append(datetime.fromtimestamp(d_ts).date().isoformat())

orders_data = {
    "order_id": list(range(1, ORDERS_ROWS + 1)),
    "customer_id": [random.randint(1, CUSTOMERS_ROWS) for _ in range(ORDERS_ROWS)],
    "order_date": order_dates,
    "amount_cents": [random.randint(100, 99999) for _ in range(ORDERS_ROWS)],
    "status": [random.choice(STATUSES) for _ in range(ORDERS_ROWS)],
}

orders_table = pa.table(orders_data)
buf = io.BytesIO()
pq.write_table(orders_table, buf)
orders_bytes = buf.getvalue()
# YOUR CODE: s3.put_object(
#   Bucket=BUCKET_NAME,
#   Key="orders/orders.parquet",
#   Body=orders_bytes,
# ).
print(f"Orders Parquet uploaded ({ORDERS_ROWS} rows, {len(orders_bytes)} bytes)")

# 2. Create the empty tables in Redshift. First pass: unoptimized.
# YOUR CODE: run_redshift_sql("DROP TABLE IF EXISTS orders;").
# YOUR CODE: run_redshift_sql("DROP TABLE IF EXISTS customers;").
# YOUR CODE: run_redshift_sql(
#   "CREATE TABLE customers ("
#   "  customer_id INTEGER, country VARCHAR(2), tier VARCHAR(10)"
#   ");"
# ).
# YOUR CODE: run_redshift_sql(
#   "CREATE TABLE orders ("
#   "  order_id INTEGER, customer_id INTEGER, order_date DATE, "
#   "  amount_cents INTEGER, status VARCHAR(16)"
#   ");"
# ).
print("Empty tables created (unoptimized first pass).")

# 3. COPY from S3 using the COPY role attached at create_cluster time.
copy_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}"
# YOUR CODE: run_redshift_sql(
#   f"COPY customers FROM 's3://{BUCKET_NAME}/customers/' "
#   f"IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;"
# ).
# YOUR CODE: run_redshift_sql(
#   f"COPY orders FROM 's3://{BUCKET_NAME}/orders/' "
#   f"IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;"
# ).
print("COPY commands issued.")

# 4. Verify.
count_desc = run_redshift_sql("SELECT count(*) FROM orders;", with_result=True)
count_records = count_desc.get("result", {}).get("Records", [])
loaded_rows = int(count_records[0][0]["longValue"]) if count_records else 0
print(f"orders row count: {loaded_rows}")

err_desc = run_redshift_sql(
    f"SELECT count(*) FROM STL_LOAD_ERRORS "
    f"WHERE filename LIKE 's3://{BUCKET_NAME}/orders/%';",
    with_result=True,
)
err_records = err_desc.get("result", {}).get("Records", [])
error_rows = int(err_records[0][0]["longValue"]) if err_records else 0
print(f"STL_LOAD_ERRORS rows for orders: {error_rows}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: orders loaded via COPY, count(*) matches ORDERS_ROWS,
# STL_LOAD_ERRORS empty for the lab's S3 prefix.


def checkpoint_2(session):
    try:
        rd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def run_sql(sql: str) -> list:
            start = rd.execute_statement(
                ClusterIdentifier=CLUSTER_ID,
                Database=DB_NAME,
                DbUser=DB_USER,
                Sql=sql,
            )
            sid = start["Id"]
            deadline = time.time() + 90
            while time.time() < deadline:
                desc = rd.describe_statement(Id=sid)
                status_ = desc["Status"]
                if status_ in ("FINISHED", "FAILED", "ABORTED"):
                    if status_ != "FINISHED":
                        raise RuntimeError(
                            f"{sid} ended {status_}: {desc.get('Error', '(no error)')}"
                        )
                    if desc.get("HasResultSet"):
                        res = rd.get_statement_result(Id=sid)
                        return res.get("Records", [])
                    return []
                time.sleep(1)
            raise RuntimeError(f"{sid} did not finish within 90s.")

        try:
            records = run_sql("SELECT count(*) FROM orders;")
        except RuntimeError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Cannot SELECT from orders: {e}. Confirm the COPY command "
                    f"ran in Task 2 and that the orders table exists in the "
                    f"{DB_NAME!r} database."
                ),
            )
        row_count = int(records[0][0]["longValue"]) if records else 0
        if row_count != ORDERS_ROWS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"orders row count is {row_count}, expected {ORDERS_ROWS}. "
                    f"If 0, COPY did not run or wrote to a different table. "
                    f"If non-zero but off, the seed write landed under a "
                    f"different S3 prefix and a partial COPY succeeded."
                ),
            )

        records = run_sql(
            f"SELECT count(*) FROM STL_LOAD_ERRORS "
            f"WHERE filename LIKE 's3://{BUCKET_NAME}/orders/%';"
        )
        err_count = int(records[0][0]["longValue"]) if records else 0
        if err_count != 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"STL_LOAD_ERRORS has {err_count} row(s) for the orders "
                    f"S3 prefix. Query STL_LOAD_ERRORS directly to see "
                    f"err_reason and the offending column; the most common "
                    f"cause is a Parquet schema mismatch against the "
                    f"CREATE TABLE column types."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks two things: `SELECT count(*) FROM orders` returns exactly `ORDERS_ROWS` and `SELECT count(*) FROM STL_LOAD_ERRORS WHERE filename LIKE 's3://{BUCKET}/orders/%'` returns 0. Read the failure reason. The most common miss on the first run is the COPY command running against the wrong table name or a misspelled IAM_ROLE ARN: redshift-data reports `FINISHED` on COPY even when zero rows landed, so always validate with a row count after. The next most common miss is forgetting to upload one of the Parquet objects; the `s3.put_object` for `orders/orders.parquet` is the one that matters here.

</details>

<details><summary>Hint 2 (direction)</summary>

Two `s3.put_object` calls (`customers/customers.parquet` and `orders/orders.parquet`). Four `run_redshift_sql` calls for `DROP TABLE` (twice, defensive) plus `CREATE TABLE` (twice). Two `run_redshift_sql` calls for `COPY` (customers first, orders second; if orders fails on a foreign-key-style lookup, the smaller customers load isolates the issue). The COPY syntax is `COPY <table> FROM 's3://<bucket>/<prefix>/' IAM_ROLE '<arn>' FORMAT AS PARQUET;` with the role ARN inline (single-quoted, since it lives inside the SQL). If the COPY command errors with `S3ServiceException: Access Denied`, the COPY role's inline policy is missing `s3:GetObject` plus `s3:ListBucket` on the bucket and `bucket/*`; re-run `iam.put_role_policy` with the policy from Task 1.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="customers/customers.parquet",
    Body=customers_bytes,
)
s3.put_object(
    Bucket=BUCKET_NAME,
    Key="orders/orders.parquet",
    Body=orders_bytes,
)

run_redshift_sql("DROP TABLE IF EXISTS orders;")
run_redshift_sql("DROP TABLE IF EXISTS customers;")
run_redshift_sql(
    "CREATE TABLE customers ("
    "  customer_id INTEGER, country VARCHAR(2), tier VARCHAR(10)"
    ");"
)
run_redshift_sql(
    "CREATE TABLE orders ("
    "  order_id INTEGER, customer_id INTEGER, order_date DATE, "
    "  amount_cents INTEGER, status VARCHAR(16)"
    ");"
)

run_redshift_sql(
    f"COPY customers FROM 's3://{BUCKET_NAME}/customers/' "
    f"IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;"
)
run_redshift_sql(
    f"COPY orders FROM 's3://{BUCKET_NAME}/orders/' "
    f"IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;"
)
```

If `redshift-data execute_statement` returns `BadRequestException: ClusterIdentifier ... is not authorized`, the `labrun-test` user is missing `redshift-data:*` (covered by `AmazonRedshiftFullAccess`); confirm the managed policy is attached. If COPY ends FAILED with `Spectrum Scan Error: S3ServiceException: Access Denied`, the COPY role's inline policy is wrong; re-run `iam.put_role_policy`. If the orders row count is 0 but COPY reported `FINISHED`, the Parquet file landed at a different key (for example `orders.parquet` at the bucket root instead of under `orders/`); list the bucket to confirm and re-upload.

</details>


**Wallet check.** Cluster meter still running at $0.25 per hour. The COPY itself is free (Redshift does not bill for COPY operations; it bills for cluster node-hours). The S3 PUT requests for the two Parquet objects are well inside the always-free tier. STL_LOAD_ERRORS reads are queries against system tables, which are free at this dataset size. The only line item that materially registers all session is the cluster node-hour rate.


## Task 3: Recreate `orders` with the right distkey and sortkey; verify the EXPLAIN deltas

The first-pass `orders` table has no distkey and no sortkey. Redshift defaults to `DISTSTYLE AUTO` (the planner picks between EVEN and ALL) and no sort order. Run the three representative analyst queries against that shape and the join-heavy one shows `DS_BCAST_INNER` or `DS_DIST_BOTH` in the plan: the planner is redistributing or broadcasting one side of the join because no distkey aligns the two tables. That redistribution is the cost driver Checkpoint 4 will catch as a slow query.

The three representative queries (the ones the BI dashboard runs):

```sql
-- Q1: filter by customer (per-customer order history page in the BI tool)
SELECT order_id, order_date, amount_cents FROM orders WHERE customer_id = 42 ORDER BY order_date DESC;

-- Q2: date-range aggregation (the BI dashboard's monthly revenue tile)
SELECT date_trunc('month', order_date) AS month, sum(amount_cents) AS rev
FROM orders WHERE order_date BETWEEN DATE '2024-04-01' AND DATE '2024-06-30'
GROUP BY 1 ORDER BY 1;

-- Q3: join orders to customers (top-N customers by revenue tile)
SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.tier ORDER BY rev DESC;
```

The right shape for these three: `DISTKEY(customer_id)` so the join in Q3 colocates rows on `customer_id` (both tables distributed the same way, no `DS_BCAST_INNER` or `DS_DIST_BOTH`), and `SORTKEY(order_date)` so Q2's range scan can prune blocks via zone maps. `dc2.large` does NOT support `ALTER TABLE ... ALTER DISTKEY` (that is an RA3-only feature); the lab walks the drop-and-recreate path: drop `orders`, recreate with the keys, COPY again from the same Parquet, then run `VACUUM` plus `ANALYZE` so the planner has fresh statistics for the EXPLAIN comparison.

Checkpoint 3 confirms `SVV_TABLE_INFO` reports `diststyle=KEY`, `dist_key=customer_id`, and the sort key includes `order_date`, AND that the EXPLAIN plan for Q3 no longer shows `DS_BCAST_INNER` or `DS_DIST_BOTH`.


In [ ]:
# NBVAL_SKIP
# Task 3: Drop and recreate orders with DISTKEY(customer_id) and
# SORTKEY(order_date), re-COPY, VACUUM plus ANALYZE, then verify
# SVV_TABLE_INFO and the EXPLAIN plan for the join query.

# Optional: run EXPLAIN against the current (unoptimized) join query so the
# student can compare. The before-plan typically shows DS_BCAST_INNER or
# DS_DIST_BOTH; the after-plan should show DS_DIST_NONE on both sides.
before = run_redshift_sql(
    "EXPLAIN SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o "
    "JOIN customers c ON o.customer_id = c.customer_id "
    "GROUP BY c.tier ORDER BY rev DESC;",
    with_result=True,
)
before_records = before.get("result", {}).get("Records", [])
before_plan = "\n".join(r[0].get("stringValue", "") for r in before_records)
print("BEFORE plan (unoptimized orders table):")
print(before_plan)
print()

copy_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{COPY_ROLE_NAME}"

# YOUR CODE: run_redshift_sql("DROP TABLE orders;").
# YOUR CODE: run_redshift_sql(
#   "CREATE TABLE orders ("
#   "  order_id INTEGER, customer_id INTEGER, order_date DATE, "
#   "  amount_cents INTEGER, status VARCHAR(16)"
#   ") DISTSTYLE KEY DISTKEY(customer_id) SORTKEY(order_date);"
# ).
# YOUR CODE: run_redshift_sql(
#   f"COPY orders FROM 's3://{BUCKET_NAME}/orders/' "
#   f"IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;"
# ).
# YOUR CODE: run_redshift_sql("VACUUM orders;").
# YOUR CODE: run_redshift_sql("ANALYZE orders;").
print("orders recreated with DISTKEY(customer_id) SORTKEY(order_date); "
      "VACUUM + ANALYZE complete.")

# After-plan for the same query.
after = run_redshift_sql(
    "EXPLAIN SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o "
    "JOIN customers c ON o.customer_id = c.customer_id "
    "GROUP BY c.tier ORDER BY rev DESC;",
    with_result=True,
)
after_records = after.get("result", {}).get("Records", [])
after_plan = "\n".join(r[0].get("stringValue", "") for r in after_records)
print("AFTER plan (orders distributed on customer_id):")
print(after_plan)
print()

# Inspect SVV_TABLE_INFO to confirm the metadata Redshift reports for the
# recreated table.
info = run_redshift_sql(
    "SELECT diststyle, sortkey1 FROM SVV_TABLE_INFO "
    "WHERE \"table\" = 'orders';",
    with_result=True,
)
info_records = info.get("result", {}).get("Records", [])
if info_records:
    diststyle = info_records[0][0].get("stringValue", "")
    sortkey1 = info_records[0][1].get("stringValue", "")
    print(f"SVV_TABLE_INFO: diststyle={diststyle!r} sortkey1={sortkey1!r}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: orders has DISTSTYLE KEY with distkey customer_id and a
# sortkey that includes order_date; the EXPLAIN plan for the join query
# does not contain DS_BCAST_INNER or DS_DIST_BOTH.


def checkpoint_3(session):
    try:
        rd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def run_sql(sql: str) -> list:
            start = rd.execute_statement(
                ClusterIdentifier=CLUSTER_ID,
                Database=DB_NAME,
                DbUser=DB_USER,
                Sql=sql,
            )
            sid = start["Id"]
            deadline = time.time() + 90
            while time.time() < deadline:
                desc = rd.describe_statement(Id=sid)
                status_ = desc["Status"]
                if status_ in ("FINISHED", "FAILED", "ABORTED"):
                    if status_ != "FINISHED":
                        raise RuntimeError(
                            f"{sid} ended {status_}: {desc.get('Error', '(no error)')}"
                        )
                    if desc.get("HasResultSet"):
                        res = rd.get_statement_result(Id=sid)
                        return res.get("Records", [])
                    return []
                time.sleep(1)
            raise RuntimeError(f"{sid} did not finish within 90s.")

        # Inspect SVV_TABLE_INFO. diststyle values in Redshift system tables:
        # "EVEN", "ALL", "AUTO(...)", or "KEY(<column>)" for explicit distkey.
        records = run_sql(
            "SELECT diststyle, sortkey1 FROM SVV_TABLE_INFO "
            "WHERE \"table\" = 'orders';"
        )
        if not records:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "orders is not visible in SVV_TABLE_INFO. The recreate "
                    "in Task 3 did not run or COPY did not refill the table; "
                    "re-run the Task 3 cell."
                ),
            )
        diststyle = records[0][0].get("stringValue", "") or ""
        sortkey1 = records[0][1].get("stringValue", "") or ""

        if not diststyle.upper().startswith("KEY"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"diststyle is {diststyle!r}, expected to start with 'KEY' "
                    f"(KEY(customer_id) on dc2.large). Re-create the table "
                    f"with DISTSTYLE KEY DISTKEY(customer_id)."
                ),
            )
        if "customer_id" not in diststyle.lower():
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"distkey is not customer_id (SVV_TABLE_INFO reports "
                    f"{diststyle!r}). The join query in Q3 needs customer_id "
                    f"as the distkey so both sides of the join colocate."
                ),
            )

        if sortkey1.lower() != "order_date":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"sortkey1 is {sortkey1!r}, expected 'order_date'. "
                    f"Q2 (date-range aggregation) relies on the sortkey "
                    f"to prune blocks via the zone map."
                ),
            )

        # EXPLAIN the join query. The after-plan must not contain
        # DS_BCAST_INNER or DS_DIST_BOTH. Either one indicates the
        # planner is moving the join key across nodes at runtime, which
        # is the Q3 cost driver.
        plan_records = run_sql(
            "EXPLAIN SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o "
            "JOIN customers c ON o.customer_id = c.customer_id "
            "GROUP BY c.tier ORDER BY rev DESC;"
        )
        plan_lines = [r[0].get("stringValue", "") for r in plan_records]
        plan_text = "\n".join(plan_lines)
        if "DS_BCAST_INNER" in plan_text or "DS_DIST_BOTH" in plan_text:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"EXPLAIN for the join query still contains a redistribution "
                    f"indicator. Plan:\n{plan_text}\n"
                    f"Make sure customers also has DISTSTYLE that lets the "
                    f"planner avoid redistribution; the canonical pattern for "
                    f"a small dimension table is DISTSTYLE ALL, but for a 500-"
                    f"row dimension you can also rely on the planner choosing "
                    f"to broadcast the smaller side. If DS_BCAST_INNER persists, "
                    f"run ANALYZE on both tables so the planner has accurate "
                    f"row counts."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint walks three things: `SVV_TABLE_INFO.diststyle` starts with `KEY` and mentions `customer_id`, `SVV_TABLE_INFO.sortkey1` equals `order_date`, and the `EXPLAIN` plan for the join query contains neither `DS_BCAST_INNER` nor `DS_DIST_BOTH`. Read the failure reason. The most common miss on the first run is forgetting to re-COPY after the DROP plus CREATE: the recreated table has the right keys but zero rows, so the EXPLAIN plan is correct but Checkpoint 4 fails on the next pass. The second most common miss is forgetting `VACUUM` plus `ANALYZE`: the table has the right keys and the right rows but the planner uses stale statistics, so `DS_BCAST_INNER` can stick around until you ANALYZE.

</details>

<details><summary>Hint 2 (direction)</summary>

Five SQL statements in this task. `DROP TABLE orders;`. `CREATE TABLE orders (... ) DISTSTYLE KEY DISTKEY(customer_id) SORTKEY(order_date);`. The same `COPY orders FROM ... IAM_ROLE ... FORMAT AS PARQUET;` you ran in Task 2 (the bytes in S3 did not change; only the table metadata changed). `VACUUM orders;` to sort the newly loaded rows by `order_date`. `ANALYZE orders;` so the planner has accurate row counts and column statistics for the EXPLAIN comparison. The 500-row `customers` table is small enough that the planner can broadcast it without `DS_BCAST_INNER` appearing as a costly step; if your EXPLAIN still shows `DS_BCAST_INNER`, run `ANALYZE customers;` too.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
run_redshift_sql("DROP TABLE orders;")
run_redshift_sql(
    "CREATE TABLE orders ("
    "  order_id INTEGER, customer_id INTEGER, order_date DATE, "
    "  amount_cents INTEGER, status VARCHAR(16)"
    ") DISTSTYLE KEY DISTKEY(customer_id) SORTKEY(order_date);"
)
run_redshift_sql(
    f"COPY orders FROM 's3://{BUCKET_NAME}/orders/' "
    f"IAM_ROLE '{copy_role_arn}' FORMAT AS PARQUET;"
)
run_redshift_sql("VACUUM orders;")
run_redshift_sql("ANALYZE orders;")
run_redshift_sql("ANALYZE customers;")
```

If `DROP TABLE` errors with `cannot drop table orders because other objects depend on it`, you have a view or another table referencing `orders` from a prior run; `DROP TABLE orders CASCADE;` clears the dependents (none should exist on a clean lab run, but a re-runner who created a view will hit this). If `VACUUM` errors with `VACUUM is running on this table`, wait a minute and re-run; only one VACUUM per table can run at a time. If the EXPLAIN plan still shows `DS_BCAST_INNER` after VACUUM plus ANALYZE, the row counts may still be off in `customers`; `INSERT INTO customers SELECT * FROM customers LIMIT 0;` is a no-op that forces a statistics refresh on some Redshift versions, but the cleaner fix is to re-run `ANALYZE customers;` explicitly.

</details>


**Wallet check.** Cluster meter still running at $0.25 per hour. DROP, CREATE, COPY, VACUUM, and ANALYZE are all free operations against the running cluster. No new S3 PUT requests; the Parquet under `s3://{BUCKET}/orders/` did not change. Total session cost so far: still well under 10 cents, dominated by cluster wall-clock since `create_cluster` returned.


## Task 4: Run the three representative queries and prove each is under 1 second server-side

The three queries from Task 3 now run against the optimized `orders` table. Server-side execution time is read from the redshift-data API: when a statement reaches `FINISHED`, `describe_statement` returns a `Duration` field in nanoseconds (Redshift returns it that way; you divide by 1e6 to get milliseconds).

The 1-second SLA the BI team needs is server-side only. Network round-trip from your notebook to the cluster does not count: the dashboard is a different client with its own latency budget. What matters for the cert pattern is that the planner can do its job in under 1000 ms on `dc2.large` once the distkey, sortkey, and statistics are correct.

Three queries to run, in order:

1. **Q1 filter-by-customer**: `SELECT order_id, order_date, amount_cents FROM orders WHERE customer_id = 42 ORDER BY order_date DESC;`. With `DISTKEY(customer_id)`, the planner can route the entire query to the single node holding `customer_id = 42`.
2. **Q2 date-range aggregation**: `SELECT date_trunc('month', order_date) AS month, sum(amount_cents) AS rev FROM orders WHERE order_date BETWEEN DATE '2024-04-01' AND DATE '2024-06-30' GROUP BY 1 ORDER BY 1;`. With `SORTKEY(order_date)`, the planner prunes blocks outside the date range via the zone map.
3. **Q3 join-against-dimension**: `SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o JOIN customers c ON o.customer_id = c.customer_id GROUP BY c.tier ORDER BY rev DESC;`. With `customers` distributed for colocation (or small enough to broadcast), the join completes without redistribution.

Each query's `Duration` (server-side) is captured into `QUERY_DURATIONS_MS` (a dict, keyed `"q1"`, `"q2"`, `"q3"`). Checkpoint 4 reads from that dict and asserts each is under 1000 ms.

If any query exceeds 1 second, the hint progression points at sort-key coverage, dist-key cardinality, or `VACUUM` plus `ANALYZE` (which Task 3 should have done; re-run them and re-run the queries).


In [ ]:
# NBVAL_SKIP
# Task 4: Run Q1, Q2, Q3 server-side, capture Duration in milliseconds for
# each. Checkpoint 4 reads QUERY_DURATIONS_MS without re-running the queries.

queries = {
    "q1": (
        "SELECT order_id, order_date, amount_cents FROM orders "
        "WHERE customer_id = 42 ORDER BY order_date DESC;"
    ),
    "q2": (
        "SELECT date_trunc('month', order_date) AS month, "
        "sum(amount_cents) AS rev FROM orders "
        "WHERE order_date BETWEEN DATE '2024-04-01' AND DATE '2024-06-30' "
        "GROUP BY 1 ORDER BY 1;"
    ),
    "q3": (
        "SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o "
        "JOIN customers c ON o.customer_id = c.customer_id "
        "GROUP BY c.tier ORDER BY rev DESC;"
    ),
}

# YOUR CODE: For each (label, sql) in queries.items(), call run_redshift_sql,
# read describe payload's Duration (nanoseconds), divide by 1_000_000 to get
# milliseconds, and write the integer into QUERY_DURATIONS_MS[label]. Print
# each value as it lands. Example:
#
# for label, sql in queries.items():
#     desc = run_redshift_sql(sql)
#     duration_ms = int(desc.get("Duration", 0) / 1_000_000)
#     QUERY_DURATIONS_MS[label] = duration_ms
#     print(f"{label}: {duration_ms} ms server-side")

print("All three queries complete.")
print(f"QUERY_DURATIONS_MS: {QUERY_DURATIONS_MS}")


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: All three queries reported a server-side Duration under
# 1000 milliseconds. Reads QUERY_DURATIONS_MS populated by Task 4. If the
# dict is empty (student skipped Task 4), the checkpoint re-runs the
# queries itself.

SLA_MS = 1000


def checkpoint_4(session):
    global QUERY_DURATIONS_MS
    try:
        rd = boto3.client(
            "redshift-data",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        def run_and_time(sql: str) -> int:
            start = rd.execute_statement(
                ClusterIdentifier=CLUSTER_ID,
                Database=DB_NAME,
                DbUser=DB_USER,
                Sql=sql,
            )
            sid = start["Id"]
            deadline = time.time() + 90
            while time.time() < deadline:
                desc = rd.describe_statement(Id=sid)
                status_ = desc["Status"]
                if status_ in ("FINISHED", "FAILED", "ABORTED"):
                    if status_ != "FINISHED":
                        raise RuntimeError(
                            f"{sid} ended {status_}: {desc.get('Error', '(no error)')}"
                        )
                    return int(desc.get("Duration", 0) / 1_000_000)
                time.sleep(1)
            raise RuntimeError(f"{sid} did not finish within 90s.")

        labels = {
            "q1": (
                "SELECT order_id, order_date, amount_cents FROM orders "
                "WHERE customer_id = 42 ORDER BY order_date DESC;"
            ),
            "q2": (
                "SELECT date_trunc('month', order_date) AS month, "
                "sum(amount_cents) AS rev FROM orders "
                "WHERE order_date BETWEEN DATE '2024-04-01' AND DATE '2024-06-30' "
                "GROUP BY 1 ORDER BY 1;"
            ),
            "q3": (
                "SELECT c.tier, sum(o.amount_cents) AS rev FROM orders o "
                "JOIN customers c ON o.customer_id = c.customer_id "
                "GROUP BY c.tier ORDER BY rev DESC;"
            ),
        }

        for label, sql in labels.items():
            if label not in QUERY_DURATIONS_MS or QUERY_DURATIONS_MS[label] <= 0:
                QUERY_DURATIONS_MS[label] = run_and_time(sql)

        slow = {k: v for k, v in QUERY_DURATIONS_MS.items() if v >= SLA_MS}
        if slow:
            details = ", ".join(f"{k}={v}ms" for k, v in sorted(slow.items()))
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Queries exceeding the 1000 ms SLA: {details}. The fixes in "
                    f"order: re-run VACUUM plus ANALYZE on orders and customers; "
                    f"confirm the DISTKEY is customer_id (Checkpoint 3); confirm "
                    f"the SORTKEY is order_date (Checkpoint 3). If Q3 is slow, "
                    f"the join is still redistributing; check the EXPLAIN plan "
                    f"for DS_BCAST_INNER."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads `QUERY_DURATIONS_MS` and asserts every value is under 1000. Read the failure reason. The most common miss is forgetting to populate `QUERY_DURATIONS_MS` from the loop body; the dict stays empty and the checkpoint re-runs the queries itself, which usually succeeds the second time because Redshift caches the compiled plan. The second most common miss is reading the wrong field from `describe_statement`: `Duration` is in nanoseconds (yes, nanoseconds), so you divide by 1e6 (not 1e3) to get milliseconds.

</details>

<details><summary>Hint 2 (direction)</summary>

One loop. For each `(label, sql)` in the `queries` dict, call `run_redshift_sql(sql)`. The returned `describe_statement` payload has a `Duration` field in nanoseconds; divide by `1_000_000` to get milliseconds and write the integer into `QUERY_DURATIONS_MS[label]`. Print each value as it lands so you can see Q1, Q2, Q3 side by side and catch outliers. If any query is slow, the first thing to try is `ANALYZE orders;` plus `ANALYZE customers;` again; the second thing is to inspect the EXPLAIN plan for the slow query to see if redistribution snuck back in.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
for label, sql in queries.items():
    desc = run_redshift_sql(sql)
    duration_ms = int(desc.get("Duration", 0) / 1_000_000)
    QUERY_DURATIONS_MS[label] = duration_ms
    print(f"{label}: {duration_ms} ms server-side")
```

If a query reports `Duration=0`, the redshift-data API did not populate it; this happens when the result set is fetched before the statement reaches `FINISHED`. The provided `run_redshift_sql` already polls to `FINISHED`, so this should not occur; if it does, re-run the cell and the second pass usually populates `Duration` correctly. If all three queries exceed 1000 ms, the cluster is cold-cache: re-run the cell a second time and the compiled-plan cache plus the data-block cache will land you under SLA. If Q3 is the only slow query, the join is still redistributing; re-check Checkpoint 3 and confirm the EXPLAIN plan does not contain `DS_BCAST_INNER` or `DS_DIST_BOTH`.

</details>


**Wallet check.** Cluster meter still running at $0.25 per hour. Each query against the cluster is free; you pay for the node-hours, not the queries (the Redshift Serverless pricing model is different, but this lab is on provisioned). Total session cost is dominated by wall-clock since `create_cluster`. If you have been moving fast, you are at $0.10 to $0.20 right now; the cleanup cell below stops the meter the moment the cluster reaches `Deleting` state.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation,
# critical-first order per RESOURCE-SAFETY-SPEC Rule 2.
#
# labrun-checks 0.3.0 lacks adapters for redshift_cluster, eventbridge_rule,
# lambda_function, and security_group. The _LabAdapter below wraps
# AwsCleanupAdapter and dispatches those four types inline so cleanup works
# end-to-end without a labrun-checks release. When the package promotes
# these adapters in a future release, the wrapper can be removed and
# run_cleanup called against the default adapter.
#
# The Redshift adapter waits for ClusterNotFound (or describe to return
# False) before returning so a second cleanup invocation (atexit racing
# the explicit run) does not double-delete or block.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds Lab 06 inline resource types.

    Inlined types: redshift_cluster, eventbridge_rule, lambda_function,
    security_group. Everything else (s3_bucket, iam_role) delegates to the
    bundled AwsCleanupAdapter unchanged.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _client(self, service: str, credentials: dict):
        return boto3.client(
            service,
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "redshift_cluster":
            client = self._client("redshift", credentials)
            try:
                client.delete_cluster(
                    ClusterIdentifier=resource.id,
                    SkipFinalClusterSnapshot=True,
                )
            except ClientError as e:
                code_ = e.response["Error"]["Code"]
                if code_ in ("ClusterNotFound", "ClusterNotFoundFault"):
                    return
                if code_ == "InvalidClusterState":
                    # Cluster is already deleting from a prior invocation.
                    pass
                else:
                    raise
            # Wait for the cluster to actually be gone before returning so a
            # second cleanup invocation does not race the delete.
            wait_deadline = time.time() + 600  # 10 minutes
            while time.time() < wait_deadline:
                try:
                    client.describe_clusters(ClusterIdentifier=resource.id)
                except ClientError as e:
                    if e.response["Error"]["Code"] in ("ClusterNotFound", "ClusterNotFoundFault"):
                        return
                    raise
                time.sleep(15)
            return

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.remove_targets(
                    Rule=resource.id,
                    Ids=["1"],
                    Force=True,
                )
            except ClientError as e:
                if e.response["Error"]["Code"] not in (
                    "ResourceNotFoundException", "InternalException"
                ):
                    raise
            try:
                client.delete_rule(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "lambda_function":
            client = self._client("lambda", credentials)
            try:
                client.delete_function(FunctionName=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "ResourceNotFoundException":
                    raise
            return

        if resource.type == "security_group":
            client = self._client("ec2", credentials)
            # Resolve the SG ID by name; create_security_group returns an ID
            # but the manifest stores the group-name for human readability.
            try:
                resp = client.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [resource.id]}],
                )
                for sg in resp.get("SecurityGroups", []):
                    try:
                        client.delete_security_group(GroupId=sg["GroupId"])
                    except ClientError as e:
                        if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                            raise
            except ClientError as e:
                if e.response["Error"]["Code"] != "InvalidGroup.NotFound":
                    raise
            return

        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "redshift_cluster":
            client = self._client("redshift", credentials)
            try:
                client.describe_clusters(ClusterIdentifier=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] in ("ClusterNotFound", "ClusterNotFoundFault"):
                    return False
                raise

        if resource.type == "eventbridge_rule":
            client = self._client("events", credentials)
            try:
                client.describe_rule(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "lambda_function":
            client = self._client("lambda", credentials)
            try:
                client.get_function(FunctionName=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "ResourceNotFoundException":
                    return False
                raise

        if resource.type == "security_group":
            client = self._client("ec2", credentials)
            try:
                resp = client.describe_security_groups(
                    Filters=[{"Name": "group-name", "Values": [resource.id]}],
                )
                return bool(resp.get("SecurityGroups", []))
            except ClientError as e:
                if e.response["Error"]["Code"] == "InvalidGroup.NotFound":
                    return False
                raise

        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# Empty the S3 bucket before run_cleanup tears it down. S3 buckets cannot be
# deleted while they contain objects; the bundled s3_bucket adapter only
# deletes one page of objects at a time.
print("Emptying the S3 bucket before teardown...")
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing to cleanup.")

# Drop inline policies on the two roles before role-delete so the
# iam_role adapter does not block on attached policies.
iam_client = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
for role_name in (COPY_ROLE_NAME, SAFETY_NET_ROLE_NAME):
    try:
        listed = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in listed.get("PolicyNames", []):
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchEntity":
            print(f"Inline policy detach for {role_name} ran into: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = sum(1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False))
standard_total = len(CLEANUP_MANIFEST) - critical_total
critical_destroyed = critical_total - sum(
    1 for r in CLEANUP_MANIFEST if getattr(r, "critical", False) and r.id in failed_ids
)
standard_destroyed = standard_total - sum(
    1 for r in CLEANUP_MANIFEST if not getattr(r, "critical", False) and r.id in failed_ids
)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: $0.15 to $0.50.** The cluster node-hour is the only line item that materially registered. At $0.25 per hour, a 35-to-60-minute session runs $0.15 to $0.25 if cleanup ran clean. Worst case (kernel died, atexit ran late, the 2-hour EventBridge safety net fired) is $0.50. The catastrophic case (cleanup skipped, atexit failed, safety-net Lambda failed) would require three independent failures and is the reason the $20-per-month AWS billing alert exists. On the cert-scenario 50-million-row dataset the same pattern runs against a `dc2.large` for closer to 90 minutes (load and queries scale roughly linearly with row count), so plan $0.40 to $0.50; for production-shape data sizes, RA3 with managed storage decouples compute from storage and the math changes.


## These are not graded. They are for you.

Three questions worth sitting with before the exam.

1. You picked `DISTKEY(customer_id)` because Q3's join is on `customer_id`. Walk the trade-off. What happens to Q1 (`WHERE customer_id = 42`) when the distkey is `customer_id` versus `order_date` versus `EVEN`? What happens to Q2 (the date-range aggregation) in those same three cases? The cert wants you to reason about each query's access pattern independently: a distkey that helps one query class can hurt another, and `DISTSTYLE ALL` (full replication) is a fourth option that trades storage for join elimination on dimension tables. For each of the four distribution styles, identify one query shape where it is the clear winner.

2. The sortkey on `order_date` made Q2's date-range scan fast via zone-map block pruning. Walk through the alternatives. A compound sortkey on `(order_date, customer_id)` covers Q1 and Q2 but doubles the metadata overhead; an interleaved sortkey on `(order_date, customer_id)` spreads the benefit but slows down VACUUM significantly. On `dc2.large` the choice is usually compound; on RA3 the calculus shifts because RA3 supports `ALTER DISTKEY` and managed storage handles VACUUM differently. Identify two cases where you would deliberately skip the sortkey (tables under 10,000 rows; staging tables that get truncated weekly) and two where interleaved beats compound (BI dashboards that filter on roughly equal cardinality columns; ad-hoc analyst queries with unpredictable WHERE clauses).

3. The 2-hour EventBridge plus Lambda safety net cost about $0 to run and stopped a forgotten cluster from costing $42 over a week. Walk through the alternatives. AWS Budgets with a forecasted-spend action can shut down resources, but only at the account level (too blunt for a lab). A CloudWatch alarm on `BillingEstimatedCharges` notifies but does not delete. A second EventBridge rule firing every 30 minutes could check the cluster's `ClusterCreateTime` and delete after 2 hours; this is actually simpler than a one-shot schedule and is what most production teams do for ephemeral resources. For each, identify when it is the right tool and when the one-shot safety net is the cleaner choice.

## Exam-Style Practice

**Q1.** A data engineer loads a 500 GB orders table into a Redshift `ra3.xlplus` cluster and the analytics team's join query against a 1 GB customers dimension runs in 18 seconds. The EXPLAIN plan contains `DS_BCAST_INNER` on the customers side. The orders table has `DISTKEY(order_id)` and the customers table has no explicit distkey. Which change will produce the largest reduction in query time?

A) Change orders to `DISTSTYLE EVEN` and add `SORTKEY(customer_id)`.

B) Change customers to `DISTSTYLE ALL` and re-run the join after `VACUUM` plus `ANALYZE`.

C) Add a partition column on `order_date` to orders and re-run the join.

D) Increase the cluster from 1 node to 4 nodes and re-run the join.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. `DISTSTYLE EVEN` spreads orders rows uniformly with no co-location, so the join still redistributes one side at runtime. `SORTKEY(customer_id)` does not help a join; sortkeys help range and equality filters via the zone map, not join distribution.
- B) Right. `DISTSTYLE ALL` replicates the small (1 GB) customers table to every node, eliminating the `DS_BCAST_INNER` broadcast at query time. The replication cost is one-time at load; query-time cost drops because no row movement happens. This is the canonical pattern for small dimension tables in a star schema; the cert expects you to recognize it.
- C) Wrong. Redshift does not use partitions the way Athena or Hive do; the analog is the sortkey, and a sortkey on a different column would not affect the join.
- D) Wrong. Adding nodes scales the per-node share of the scan and the parallelism on the aggregate, but the `DS_BCAST_INNER` step still moves the full customers table to every node. The redistribution is the bottleneck; node count amplifies it, not avoids it.

</details>

**Q2.** A team runs Redshift `dc2.large` and wants to change the distribution key on an existing 200 GB table from `order_id` to `customer_id` to improve a join query. The cluster has been live for 30 days and the table is in active production use. What is the correct way to make this change?

A) `ALTER TABLE orders ALTER DISTKEY customer_id;` followed by `ANALYZE orders;`.

B) Create a new empty table with `DISTKEY(customer_id)`, `INSERT INTO new_table SELECT * FROM orders;`, drop the old table, rename the new one, then `VACUUM` plus `ANALYZE`.

C) Issue `ALTER TABLE orders SET DISTSTYLE KEY DISTKEY customer_id;` and then `VACUUM` plus `ANALYZE`.

D) Use `redshift.modify_cluster` to enable the distkey-change feature and then `ALTER TABLE orders ALTER DISTKEY customer_id;`.

<details><summary>Show answer</summary>

**B**.

- A) Wrong. `ALTER TABLE ... ALTER DISTKEY` exists in Redshift, but it is supported on RA3 node types only, not `dc2.large`. On dc2 the statement returns `ERROR: ALTER DISTKEY is not supported on this cluster type`.
- B) Right. The canonical dc2.large recipe is to recreate the table with the new keys: create a new empty table with the right `DISTKEY` and `SORTKEY`, copy the data with `INSERT INTO ... SELECT FROM`, drop the old table, rename the new one, then `VACUUM` plus `ANALYZE` to sort the data by the sortkey and refresh the planner statistics. This is what this lab walks in Task 3.
- C) Wrong. There is no `ALTER TABLE ... SET DISTSTYLE` syntax in Redshift. The keyword form is `ALTER DISTKEY` (RA3 only) or `ALTER DISTSTYLE` (also RA3 only, and the right syntax to switch styles).
- D) Wrong. `modify_cluster` is for cluster-level settings (node type, port, parameter group, security groups). There is no "enable distkey-change" toggle; the capability is tied to the node type and cannot be enabled on dc2.

</details>

**Q3.** A data engineer runs a Redshift `COPY` command to load 5 GB of Parquet from S3. The command returns `FINISHED` via the redshift-data API but `SELECT count(*)` returns zero rows on the target table. What is the correct first diagnostic step?

A) `SELECT * FROM STL_LOAD_ERRORS WHERE filename LIKE 's3://<bucket>/<prefix>/%' ORDER BY starttime DESC LIMIT 50;` and read the `err_reason` plus `colname` columns.

B) Re-run the `COPY` command with `STATUPDATE ON` so Redshift refreshes statistics during the load.

C) Run `VACUUM FULL` on the target table to reclaim space and rebuild the row count metadata.

D) Check CloudWatch Logs for the Redshift cluster's `query` log group and search for the COPY's query ID.

<details><summary>Show answer</summary>

**A**.

- A) Right. `STL_LOAD_ERRORS` is the canonical Redshift system table for failed-row diagnostics during `COPY`. Even when `COPY` reports `FINISHED`, individual rows can be rejected for type mismatches, NOT NULL violations, or Parquet schema mismatches; the table reports `err_reason` (the failure category), `colname` (the column that failed), and `raw_line` (the bad data). On a Parquet load, the most common failure is a column-type mismatch between the Parquet schema and the `CREATE TABLE` types. Reading STL_LOAD_ERRORS gives you the exact row and column.
- B) Wrong. `STATUPDATE ON` (also a `COPY` option) refreshes statistics after the load, which affects query plans but does not affect what rows landed; the row count is already what it is.
- C) Wrong. `VACUUM FULL` reclaims deleted-row space and rebuilds sort order; it does not retrieve rows that failed to load. The row count is the row count.
- D) Wrong. CloudWatch query logs are useful for slow-query diagnosis on the cluster, not for COPY load errors. STL_LOAD_ERRORS is the right table; the cert expects you to know that COPY can succeed (`FINISHED`) while losing rows.

</details>
